# November changes

# Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity


import random
from itertools import combinations,chain
import time
import copy
import math
from  scipy import sparse


from tqdm.notebook import trange, tqdm
# from tqdm import tqdm,trange


import torch
import torch.nn as nn
import torch.optim as optim

import os
import collections

import matplotlib.pyplot as plt
from functools import lru_cache

##### testing
from torch.backends import cudnn

cudnn.benchmark=True
# torch.use_deterministic_algorithms(True)

torch.__version__


# Collections

In [ ]:
Transition = collections.namedtuple('Transition',
                                    field_names=['state','next_state'])

Experience = collections.namedtuple('Experience',
                                    field_names=['state','action','reward',
                                                 'done','next_state','state_rep','next_state_rep','state_topN','state_topN_map','next_topN','next_topN_map'])

Result = collections.namedtuple('Result',
                                    field_names=['reward','penalty','cost',
                                                 'max_iter','time'])


# Utility Functions


## Numpy Utility

In [ ]:
def normalize_tensor(input_tensor):
    max=torch.max(input_tensor)
    min=torch.min(input_tensor)
    if min==max:
        return torch.ones_like(input_tensor)
    return (input_tensor-min)/(max-min)




def create_u(number_of_contents,dataset=None):
    # Create content relation matrix
    if dataset is None:
        u = np.random.uniform(0, 1, (number_of_contents, number_of_contents)).astype(np.float16)

    else:
        print("Got dataset.")
        df=pd.read_csv(dataset)
        df=df.drop('timestamp',axis='columns')
        # df.sort_values(by=['movieId'], ascending=True)
        movie_ratings_count = df.groupby('movieId')['userId'].count()
        top_500=movie_ratings_count.sort_values(ascending=False).head(number_of_contents)

        df_filtered = df[df['movieId'].isin(top_500.index)]

        rating_matrix = df_filtered.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

        sparse_matrix = csr_matrix(rating_matrix)

        u = cosine_similarity(sparse_matrix).astype(np.float16)

    np.fill_diagonal(u,0)


    return np.array(u)




def create_popularity(number_of_contents, dataset=None, uniform=True, start=0, end=-1):
    if end == -1:
        end = number_of_contents

    # Boundary checks
    if start < 0 or end > number_of_contents or start > end:
        raise ValueError("Invalid start or end range.")

    if dataset is None:
        size = end - start
        if uniform:
            # Uniform distribution
            popularity_probabilities = [0] * start + [1/size] * size + [0] * (number_of_contents - end)
        else:
            # Random distribution
            random_probs = [random.random() for _ in range(size)]
            total = sum(random_probs)
            normalized_probs = [x / total for x in random_probs]  # Normalize
            popularity_probabilities = [0] * start + normalized_probs + [0] * (number_of_contents - end)
    else:
        print("Got dataset.")
        df = pd.read_csv(dataset)
        if 'timestamp' in df.columns:
            df = df.drop('timestamp', axis='columns')
        df = df.sort_values(by=['movieId'], ascending=True)
        movie_ratings_count = df.groupby('movieId')['userId'].count()
        top_movies = movie_ratings_count.sort_values(ascending=False).head(number_of_contents)

        df_filtered = df[df['movieId'].isin(top_movies.index)]
        movie_ratings_count = df_filtered.groupby('movieId')['userId'].count()
        total_ratings = movie_ratings_count.sum()
        popularity_probabilities = (movie_ratings_count / total_ratings).tolist()

    return np.array(popularity_probabilities)

def find_action_index_from_states(cache_states,state_id,next_state_id,cache_size,id_to_cache):
    state_info=find_state_by_id_faster(cache_states,id_to_cache,state_id)
    next_state_info=find_state_by_id_faster(cache_states,id_to_cache,next_state_id)
    index=0
    for i in state_info[1]:
        if i not in next_state_info[1]:
            break
        index+=1
    else:
        index=cache_size
    # index of action in q-table
    return (next_state_info[0]*(cache_size+1)) +index


def action_format_pi(cache_size,current_state,action_index):
    cache_action= action_index % (cache_size+1)
    content_action = action_index // (cache_size+1)
    state=content_action,tuple(generate_cache_state(cache_size,current_state,cache_action))

    return state

def action_format(cache_size,current_state,action_index,topN):
    cache_action= action_index % (cache_size+1)
    content_action_index = action_index // (cache_size+1)
    content_action=topN[content_action_index]

    state=content_action,generate_cache_state(cache_size,current_state,cache_action)

    return state


def find_all_states(number_of_contents,cache_size):

    # all possible cache states
    cached_combinations = list(combinations(range(0, number_of_contents), cache_size))

    #dictionary whit all possible states(current + cache)
    state_space = {}
    state_index = 0
    for content in range(0, number_of_contents):
        for cached_contents in cached_combinations:
            state_space[(content, cached_contents)] = state_index
            state_index += 1

    return   state_space

def find_all_cache_states(number_of_contents,cache_size):

    # all possible cache states
    cached_combinations = list(combinations(range(0, number_of_contents), cache_size))

    cache_state = {}
    cache_index = 0

    for cache_contents in cached_combinations:
        cache_state[cache_contents] = cache_index
        cache_index+=1

    return cache_state


def find_state_by_id_faster(cache_states,id_to_cache,state_id):
    n = len(cache_states)
    content = state_id // n
    cache = id_to_cache[state_id % n]
    return content, cache


def check_cache(cache1,cache2):
   # Checks if the cache difference is less than or equal to one content.
     return len(set(cache1) - set(cache2)) <= 1


def check_action_PI(cache_states,cache_size,current_state_id,next_state_id,action_idx,id_to_cache,):
    cache_action= action_idx % (cache_size+1)
    current_state=find_state_by_id_faster(cache_states,id_to_cache,current_state_id)
    next_state=find_state_by_id_faster(cache_states,id_to_cache,next_state_id)
    if(next_state[0]==current_state[0]):
      return False
    else:
      if(current_state[1]==next_state[1] and cache_action==cache_size): # not save same cache
          return True
      elif(check_cache(current_state[1],next_state[1]) and current_state[1][cache_action] not in next_state[1]  and current_state[0] in next_state[1] ): # save, max cache difference one content (the currelnty watched)
          return True
      return False



def check_action(cache_states,current_state_id,id_to_cache,next_state_id):
   # Returns True if eligible action, else False.
   current_state=find_state_by_id_faster(cache_states,id_to_cache,current_state_id)
   next_state=find_state_by_id_faster(cache_states,id_to_cache,next_state_id)
   if(next_state[0]==current_state[0]):
      return False
   else:
        if(current_state[1]==next_state[1]): # not save same cache
            return True
        elif(check_cache(current_state[1],next_state[1]) and current_state[0] in next_state[1] ): # save, max cache difference one content (the currelnty watched)
            return True
        return False

def check_action_DQN(current_state,next_state):
    """
    Returns True if eligible action, else False.
    """
    if(next_state[0]==current_state[0]):
        return False
    else:
        if(current_state[1]==next_state[1]): # not save same cache
            return True
        elif(check_cache(current_state[1],next_state[1]) and current_state[0] in next_state[1] ): # save, max cache difference one content (the currelnty watched)
            return True
        return False

def generate_random_cache_state(cache_size, current_state):
    """
    Creates an eligible random state of the cache.
    """
    previous_cache = set(current_state[1])
    cache = list(previous_cache)
    index_to_change = random.randint(-1, cache_size-1)
    if(index_to_change!=-1 and current_state[0] not in cache):
        cache[index_to_change] = current_state[0]
    return sorted(cache)

def generate_cache_state(cache_size, current_state,index_to_change):
    # Creates an eligible random state of the cache
    previous_cache = set(current_state[1])
    cache = list(previous_cache)
    if(index_to_change!=cache_size and current_state[0] not in cache):
        cache[index_to_change] = current_state[0]
    return sorted(cache)



def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic=True
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


## Plot Utility

In [ ]:

def graphic_compare_results_reward(list_of_results,eps_decay,title=None,label=None,episode_slices=1000,file_name=None):

    for j in range(len(list_of_results)):
        reward_list=[]
        for i in range(list_of_results[j].max_iter//episode_slices):
            reward_list.append(100*np.mean(list_of_results[j].reward[i*episode_slices:(i+1)*episode_slices]))
        # reward_list_100=[i*100 for i in reward_list]
        if(label is not None):
            plt.plot([i*episode_slices for i in range(int(list_of_results[j].max_iter/episode_slices))],reward_list,label=label[j])
        else:
            plt.plot([i*episode_slices for i in range(int(list_of_results[j].max_iter/episode_slices))],reward_list,label=str(j+1))

    if title is not None:
        plt.title(title)
    else:
        plt.title("Cache hit rate")
    plt.legend()
    plt.xlabel("Episodes")
    plt.ylabel("Cache hit rate per episode %")
    plt.ylim((0,100))
    plt.grid()
    if file_name is not None:
        plt.savefig(file_name)
    plt.show()



def graphic_compare_results_cost(list_of_results,eps_decay,title=None,label=None,episode_slices=1000,file_name=None):

    for j in range(len(list_of_results)):
        costs_list=[]
        for i in range(list_of_results[j].max_iter//episode_slices):
            costs_list.append(np.mean(list_of_results[j].cost[i*episode_slices:(i+1)*episode_slices]))
        if(label is not None):
            plt.plot([i*episode_slices for i in range(int(list_of_results[j].max_iter/episode_slices))],costs_list,label=label[j])
        else:
            plt.plot([i*episode_slices for i in range(int(list_of_results[j].max_iter/episode_slices))],costs_list,label=str(j+1))

    if title is not None:
        plt.title(title)
    else:
        plt.title("Q-value Update Difference")
    plt.legend()
    plt.xlabel("Episodes")
    plt.ylabel("MSE Loss")
    # plt.ylim((0,1))
    plt.grid()
    if file_name is not None:
        plt.savefig(file_name)
    plt.show()





def reward_plotting(list_of_rewards,episodes,title=None,label=None,episode_slices=1000):

    reward_list=[]
    for i in range(len(list_of_rewards)//episode_slices):
        reward_list.append(np.mean(list_of_rewards[i*episode_slices:(i+1)*episode_slices]))
    if(label is not None):
        plt.plot([i*episode_slices for i in range(len(list_of_rewards)//episode_slices)],reward_list,label=label[0])
    else:
        plt.plot([i*episode_slices for i in range(len(list_of_rewards)//episode_slices)],reward_list,label=str(0))

    if title is not None:
        plt.title(title)
    else:
        plt.title("Reward")
    plt.legend()
    plt.xlabel("Episodes")
    plt.ylabel("CHR %")
    plt.grid()
    plt.show()



## Training Utility

In [ ]:
def _periodic_plot(episodes,plot_interval,rewards,costs,extra_lists,extra_titles,episode_slices,threshold,early_stopage=False):
    """
    Plot diagnosticsevery plot_interval episodes and check early stopping.

    Return True of training should stop eraly, False otherwise.
    """

    try :
        reward_plotting(rewards,episodes,title="Cache Hit Rate",label=None,episode_slices=500)

        if extra_lists:
            for lst, title in zip(extra_lists,extra_titles):
                reward_plotting(lst,episodes,title,label=None,episode_slices=episode_slices)


        if costs:
            reward_plotting(costs,episodes,title="Loss",label=None,episode_slices=episode_slices)
    except Exception:

        pass # plotting is optinal never stop training is plotting fails

    if threshold> 0 and episodes>= 2 * plot_interval  and early_stopage==True:
        delta=np.abs(np.mean(rewards[-plot_interval:]) - np.mean(rewards[-2*plot_interval:-plot_interval]))
        if delta < threshold :
            print(f"Converged (Δ={delta:.6f}< {threshold}). Stopping")
            return True


    return False

## Cleanup Utility


In [ ]:
def finalize_training(agent):
    # once training is done we don't need most of this anymore, just freeing up memory/vram
    del agent.memory
    agent.memory = None

    del agent.q_network_target
    agent.q_network_target = None

    del agent.optimizer
    del agent.scheduler
    agent.optimizer = None
    agent.scheduler = None

    del agent.scaler
    agent.scaler = None

    import gc
    gc.collect()

    if agent.device.type == "cuda":
        torch.cuda.empty_cache()

    # freeze the policy network, we only need it for inference now
    agent.q_network_policy.eval()
    for param in agent.q_network_policy.parameters():
        param.requires_grad = False

    print("Training cleanup complete.")


# Environemnt

In [ ]:
# this Environment class replaces the 5-6 separate env classes I had before
# (Environment_PI, Environment_without_caching, Environment_with_caching, Environment_DQN_NC/WC, Non_RL_env)
# they were all basically the same thing with small differences, so I merged them.
#
# tabular=True  -> builds the full state table, used for Policy Iteration / Q-learning (only works for small N)
# tabular=False -> tuple states (content_id, cache_list), used for DQN / non-RL agents, scales to N=5000 etc.
#
# note: cache is always kept as a plain list (not tuple) in tuple mode so agent code can do state[1]+[x] etc.

# Environment_PI = Environment                 # tabular=True
# Environment_without_caching = Environment    # tabular=True
# Environment_with_caching = Environment       # tabular=True
# Environment_DQN_NC = Environment             # tabular=False (default)
# Environment_DQN_WC = Environment             # tabular=False (default)
# Non_RL_env = Environment                     # tabular=False (default)


def _choose_random_state(popularity):
    return np.random.choice(len(popularity), p=popularity)
    


def _check_cache(cache1, cache2):
    # true if the two caches differ by at most one item
    return sum(1 for c in cache1 if c not in cache2) <= 1


def _check_action_tabular(cache_states, cache_size, cur_id, nxt_id, id_to_cache):
    current_state=_state_by_id(cache_states,id_to_cache,cur_id)
    next_state=_state_by_id(cache_states,id_to_cache,nxt_id)
    if(next_state[0]==current_state[0]):
        return False
    else:
          if(current_state[1]==next_state[1]): # not save same cache
              return True
          elif(_check_cache(current_state[1],next_state[1]) and current_state[0] in next_state[1] ): # save, max cache difference one content (the currelnty watched)
              return True
          return False




def _state_by_id(cache_states, id_to_cache,state_id):
    # decode integer state id back into (content, cache_tuple)
    n = len(cache_states)
    return state_id // n, id_to_cache[state_id % n]



def _build_all_states(number_of_contents, cache_size):
    # builds {(content, cache_tuple): state_id} and {cache_tuple: cache_id}, only used in tabular mode
    combos = list(combinations(range(number_of_contents), cache_size))
    cache_states = {c: i for i, c in enumerate(combos)}
    all_states = {}
    sid = 0
    for content in range(number_of_contents):
        for cache in combos:
            all_states[(content, cache)] = sid
            sid += 1
    return all_states, cache_states


def _random_tuple_state(number_of_contents, cache_size):
    content = random.randrange(number_of_contents)
    cache = sorted(random.sample(range(number_of_contents), cache_size))  # list, not tuple
    return content, cache


class Environment:
    # number_of_contents, cache_size: catalogue / cache sizes
    # rewards: [miss_reward, hit_reward]
    # prob_follow: threshold/probability the user follows a recommendation
    # prob_leave: probability the user leaves the session each step
    # user_type: 'random' or 'quality_aware'
    # dataset: path to ratings csv, if None u/popularity are generated randomly
    # u, popularity: precomputed similarity matrix / popularity vector, if you already have them
    # tabular: True enumerates the whole state space (needed for PI/Q-learning), False is the DQN/non-RL mode (default)

    def __init__(
        self,
        number_of_contents,
        cache_size,
        rewards,
        number_of_recommendations,
        prob_follow,
        prob_leave,
        user_type,
        dataset=None,
        u=None,
        popularity=None,
        tabular=False,
    ):
        self.number_of_contents      = number_of_contents
        self.cache_size              = cache_size
        self.rewards                 = rewards
        self.number_of_recommendations = number_of_recommendations
        self.prob_follow             = prob_follow
        self.probability_to_follow_recommendation = prob_follow  # alias, some old loops still use this name
        self.prob_leave              = prob_leave
        self.user_left               = prob_leave    # alias, some old loops still use this name
        self.user_type               = user_type
        self.tabular                 = tabular

        # similarity matrix
        if u is not None:
            self.u = u
        else:
            self.u=create_u(number_of_contents,dataset)


        # popularity
        if popularity is not None:
            self.popularity = popularity
        else:
            self.popularity = create_popularity(number_of_contents,dataset,uniform=False)

        # popularity with the current content zeroed out and renormalized, precomputed per content
        self._pop_excluding = []
        for c in range(number_of_contents):
            p = self.popularity.copy().astype(np.float64)
            p[c] = 0.0
            p /= p.sum()
            self._pop_excluding.append(p)

        # precompute which contents are "good enough" recommendations for each content
        self.recommended = [
            np.where(self.u[c] >= self.prob_follow)[0]
            for c in range(number_of_contents)
        ]

        # only needed for tabular mode
        if tabular:
            self.all_states, self.cache_states = _build_all_states(
                number_of_contents, cache_size
            )
            self.number_of_states  = len(self.all_states)
            self.number_of_actions = self.number_of_states
            self.id_to_cache       = { idx: cache for cache,idx in self.cache_states.items()}
            self.reward_matrix     = self._build_reward_matrix()


    def _build_reward_matrix(self):
        # reward_matrix[state_id][content] = 1 if content is cached (and isn't the one being watched), else 0
        # tabular mode only
        matrix = np.zeros((self.number_of_states, self.number_of_contents), dtype=np.float32)
        for sid in range(self.number_of_states):
            content, cache = _state_by_id(self.cache_states,self.id_to_cache, sid)
            for cached_item in cache:
                if cached_item != content:
                    matrix[sid][cached_item] = 1.0
        return matrix

    def _load_u_from_dataset(self, dataset):
        # builds the similarity matrix from a ratings csv using cosine similarity
        df = pd.read_csv(dataset)
        if 'timestamp' in df.columns:
            df = df.drop('timestamp', axis=1)
        top_items = (
            df.groupby('movieId')['userId']
            .count()
            .sort_values(ascending=False)
            .head(self.number_of_contents)
            .index
        )
        df = df[df['movieId'].isin(top_items)]
        rating_matrix = df.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)
        sparse = csr_matrix(rating_matrix)
        u = cosine_similarity(sparse).astype(np.float32)
        return u



    def is_cached(self, cache, content):
        return content in cache

    def calculate_reward(self, current_state, next_state):
        # works for both modes: tabular states are ints, tuple states are (content, cache_tuple)
        if current_state is None or next_state is None:
            return None

        if self.tabular:
            _, cache = _state_by_id(self.cache_states,self.id_to_cache, current_state)
            next_content, _ = _state_by_id(self.cache_states,self.id_to_cache, next_state)
        else:
            cache = current_state[1]
            next_content = next_state[0]

        return self.rewards[1] if self.is_cached(cache, next_content) else self.rewards[0]

    def calculate_reward_faster(self, current_state, next_state):
        # matrix lookup version, tabular mode only, kept for the old training loops
        if not self.tabular:
            raise RuntimeError("calculate_reward_faster is only available in tabular mode.")
        next_content = _state_by_id(self.cache_states,self.id_to_cache, next_state)[0]
        return self.reward_matrix[current_state][next_content]

    def simulate(self, action, state):
        # simulates one user step and returns new_state, reward, done
        # action is a single-element list: [int] in tabular mode, [(content_id, cache_tuple)] in tuple mode
        done = random.random() < self.prob_leave

        if self.tabular:
            return self._simulate_tabular(action, state, done)
        else:
            return self._simulate_tuple(action, state, done)

    def _simulate_tabular(self, action, state, done):
        # tabular version, states are integer ids
        action_info = _state_by_id(self.cache_states,self.id_to_cache, random.choice(action))
        state_info  = _state_by_id(self.cache_states,self.id_to_cache,state)

        if np.mean(self.u[state_info[0], action_info[0]]) > self.prob_follow:
            new_state = random.choice(action)
        else:
            new_state_info = (
                _choose_random_state(self.popularity),
                action_info[1],
            )
            new_state = self.all_states[new_state_info]
            while not _check_action_tabular(
                self.cache_states, self.cache_size,
                state, new_state,
                self.id_to_cache
            ):
                new_state_info = (
                    _choose_random_state(self.popularity),
                    action_info[1],
                )
                new_state = self.all_states[new_state_info]

        reward = self.calculate_reward_faster(state, new_state)
        return new_state, reward, done

    def _simulate_tuple(self, action, state, done):
        # tuple version, states are (content_id, list[int])
        # cache is always turned into a list here regardless of what the agent passed in
        action_to_take = random.choice(action)
        cache = list(action_to_take[1])

        if self.user_type == 'random':
            if (random.random() < self.prob_follow
                    and action_to_take[0] != state[0]):
                new_state = action_to_take[0], cache
            else:
                new_state = int(np.random.choice(self.number_of_contents, p=self._pop_excluding[state[0]]))

        elif self.user_type == 'quality_aware':
            if np.mean(self.u[state[0], action_to_take[0]]) > self.prob_follow:
                new_state = action_to_take[0], cache
            else:
               new_state = int(np.random.choice(self.number_of_contents, p=self._pop_excluding[state[0]])),cache

        else:
            raise ValueError(f"Unknown user_type '{self.user_type}'. "
                             "Use 'random' or 'quality_aware'.")

        reward = self.calculate_reward(state, new_state)

        return new_state, reward, done

    def refresh(self):
        # returns a random starting state: int in tabular mode, (content_id, cache_tuple) otherwise
        if self.tabular:
            return random.randrange(self.number_of_states)
        return _random_tuple_state(self.number_of_contents, self.cache_size)


# Policy Iteration

## Policy Iteration Agent

In [ ]:

class Agent_PI:
    def __init__(self,env,number_of_contents,cache_size,probability_to_follow_recommendation,u=None,dataset=None,popularity=None,gamma=0.95,max_iter=5000,threshold=1e-4):

        self.gamma=gamma
        self.number_of_contents=number_of_contents
        self.cache_size=cache_size
        self.threshold=threshold
        self.env=env
        if u is None:
            self.u=create_u(self.number_of_contents)
        else:
            self.u=u

        if popularity is None:

            self.popularity=create_popularity(self.number_of_contents,dataset)
        else:
            self.popularity=popularity

        self.all_states=find_all_states(self.number_of_contents,self.cache_size)
        self.cache_states=find_all_cache_states(self.number_of_contents,self.cache_size)
        self.number_of_states=self.all_states.__len__()
        self.number_of_actions=self.number_of_contents*(self.cache_size+1)
        self.probability_to_follow_recommendation=probability_to_follow_recommendation
        self.max_iter=max_iter
        self.reward_matrix=self.create_reward_matrix()
        self.transition_matrix=self.create_transition_matrix()

        self.policy,self.value=self.PI_train()


    def create_reward_matrix(self):
        reward_matrix=np.zeros([self.number_of_states,self.number_of_contents])
        for i in range(len(self.all_states)):
            state=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,i)
            for j in state[1]:
                reward_matrix[i][j]=1
                reward_matrix[i][state[0]]=0
        return reward_matrix

    def create_transition_matrix(self):
        transition_matrix=np.empty((self.number_of_states,self.number_of_actions),dtype=object)
        for i in range(len(self.all_states)):
            state=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,i)
            for j in range(self.number_of_actions):
                next_state_list=[]
                next_state=action_format_pi(self.cache_size,state,j)
                next_state_id=self.all_states[next_state]
                if(check_action_PI(self.cache_states,self.cache_size,i,next_state_id,j,self.env.id_to_cache)==True):
                    if( self.u[state[0]][next_state[0]]>self.probability_to_follow_recommendation):
                        next_state_list.append(next_state_id)
                        transition_matrix[i][j] = [[1.0] , [self.reward_matrix[i][next_state[0]]] ,next_state_list]
                    else:
                        probs=[]
                        rewards=[]
                        for k in range(self.number_of_states):
                            possible_state=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,k)
                            if  state[0]!=possible_state[0] and next_state[1]==possible_state[1]:
                               next_state_list.append(k)
                               probs.append(self.popularity[possible_state[0]])
                               rewards.append(self.reward_matrix[i][possible_state[0]])


                        transition_matrix[i][j]=[probs,rewards,next_state_list]

                else:
                    continue

        return transition_matrix




    def PI_train(self):

        policy=np.random.choice(self.number_of_actions,self.number_of_states)

        while True:

            # policy evaluation
            V=np.zeros(self.number_of_states)
            for _ in range(self.max_iter):
                delta=0
                for s in range(self.number_of_states):
                    v=V[s]
                    action=policy[s]
                    if(self.transition_matrix[s][action] is None):
                        continue
                    prob,reward,next_state=self.transition_matrix[s][action]
                    V[s]= sum(prob[i] * (reward[i] + self.gamma * V[next_state[i]]) for i in range(len(prob)))

                    delta=max(delta,abs(v-V[s]))
                if(delta < self.threshold):
                    break

            # policy improvement
            policy_stable=True
            for s in range(self.number_of_states):
                old_action=policy[s]
                action_values=np.zeros(self.number_of_actions)
                for action in range(self.number_of_actions):
                    if(self.transition_matrix[s][action] is None):
                        continue
                    prob,reward,next_state=self.transition_matrix[s][action]
                    action_values[action]= sum(prob[i] * (reward[i] + self.gamma * V[next_state[i]]) for i in range(len(prob)))
                policy[s]=np.argmax(action_values)
                if old_action != policy[s]:
                    policy_stable=False
            if policy_stable:
                break

        print("Optimal Policy:",policy)
        print("optimal Value Function",V)

        return policy,V

    def PI_run(self,state_id):
        act=[]
        action=self.policy[state_id]
        state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state_id)
        next_state_info=action_format_pi(self.cache_size,state_info,action)
        next_state_id=self.all_states[next_state_info]

        act.append(next_state_id)

        return act








## Policy Iteration Update Function

In [ ]:
def update_PI(env,agent,max_iter = 1000,plot_interval=2000,threshold = 0.1):


    episodes=0
    max_difference=threshold+1
    start_time = time.time()

    list_of_rewards = []
    list_of_penalties = []

    while ( episodes < max_iter):


        state = env.refresh()
        done = False
        i=0
        current_penalty = 0
        current_reward = 0

        while (not done):
            action=agent.PI_run(state)

            next_state,reward,done=env.simulate(action,state)

            state=next_state

            i+=1
            current_reward+=reward

            if reward == env.rewards[0]:
                current_penalty+=1

        episodes+=1

        if(i!=0):
            list_of_penalties.append(current_penalty/i)
            list_of_rewards.append(current_reward/i)

        if(episodes % plot_interval == 0):
            print('Episodes : ',episodes, ' / ' , max_iter)
            _periodic_plot(episodes,plot_interval,list_of_rewards,costs=None,
                           extra_lists=None, extra_titles=None, episode_slices=500,
                           threshold=threshold,early_stopage=False)



    running_time=time.time()-start_time

    return running_time,episodes,list_of_rewards,list_of_penalties





# Q-Learning

## Q-Learning without caching decisions

### Q-Learning Agent without caching decisions

In [ ]:
class Agent_without_caching:
# The agent class simulates the system
    def __init__(self,env,learning_rate=0.1,gamma=0.8,eps=0.9,max_iter=50000):

        self.learning_rate=learning_rate
        self.gamma=gamma
        self.number_of_states=env.number_of_states
        self.all_states=env.all_states
        self.cache_states=env.cache_states
        self.number_of_contents=env.number_of_contents
        self.cache_size=env.cache_size
        self.env=env

        self.q_table=np.zeros([self.number_of_states,self.number_of_contents])

        # E-greedy parameters
        self.eps_init=eps
        self.eps=self.eps_init
        self.eps_decay_rate=5*1e-5
        self.eps_min=0.05

    def choose_actions(self,state):
        action=[]

        if random.uniform(0,1) < self.eps:
            #Explore state space
            state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state)
            next_state_info=random.randint(0,self.number_of_contents-1),tuple(state_info[1])
            next_state=self.all_states[next_state_info]
            while(state_info[0]==next_state_info[0]):
                next_state_info=random.randint(0,self.number_of_contents-1),tuple(state_info[1])
                next_state=self.all_states[next_state_info]
        else:
            #Exploit learned values
            i=1
            state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state)
            top_k = np.argpartition(self.q_table[state], -self.number_of_contents)[-10:]
            action_index = top_k[np.argsort(self.q_table[state][top_k])[-i]]
            next_state_info=action_index,tuple(state_info[1])
            next_state=self.all_states[next_state_info]
            while(state_info[0]==next_state_info[0]):
                i+=1
                action_index=np.argsort(self.q_table[state])[-i]
                next_state_info=action_index,tuple(state_info[1])
                next_state=self.all_states[next_state_info]

          # baseline policy(caching most popular)

        if self.env.popularity[state_info[0]]>np.min(self.env.popularity[list(state_info[1])]) and state_info[0] not in state_info[1]:
            index=np.argmin(self.env.popularity[list(state_info[1])])
            cache=generate_cache_state(self.env.cache_size,state_info,index)
            next_state_info=next_state_info[0],tuple(cache)
            next_state=self.all_states[next_state_info]

        action.append(next_state)

        return action


    def learn(self,state,action,reward,next_state):
        new_value=0
        old_value=0
        for act in action:
        # TO DO
            action_index=find_action_index_from_states(self.cache_states,state,act,self.cache_size,self.env.id_to_cache)
            action_index=find_state_by_id_faster(self.cache_states,act,self.env.id_to_cache)[0]
            old_value=self.q_table[state,action_index]
            next_state_max=np.max(self.q_table[next_state])
            new_value=(1-self.learning_rate)*old_value + self.learning_rate * ( reward + self.gamma * next_state_max)
            self.q_table[state,action_index] = new_value
        return new_value,old_value

    def eps_decay(self,episode):
        self.eps=self.eps_min +(self.eps_init-self.eps_min)*math.exp(-episode*self.eps_decay_rate)
        return



## Q-Learning with caching decisions

### Q-Learning Agent capable of making caching decisions

In [ ]:
class Agent_with_caching:
# The agent class simulates the system that takes the reccomendations and caching decisions
    def __init__(self,env,learning_rate=0.1,gamma=0.8,eps=0.9,max_iter=50000):
        self.env=env
        self.learning_rate=learning_rate
        self.gamma=gamma
        self.number_of_states=self.env.number_of_states
        self.all_states=self.env.all_states
        self.cache_states=self.env.cache_states
        self.number_of_contents=self.env.number_of_contents
        self.cache_size=self.env.cache_size

        self.q_table=np.zeros([self.number_of_states,self.number_of_contents*(self.cache_size+1)])

        # E-greedy parameters
        self.eps_init=eps
        self.eps=self.eps_init
        self.eps_decay_rate=5*1e-5
        self.eps_min=0.05

    def choose_actions(self,state):
        action=[]

        if random.uniform(0,1) < self.eps:
            #Explore state space
            state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state)
            next_state_info=(random.randint(0,self.number_of_contents-1),tuple(generate_random_cache_state(self.cache_size,state_info)))
            next_state=self.all_states[next_state_info]
            while(check_action(self.cache_states,state,self.env.id_to_cache,next_state)== False):
                next_state_info=(random.randint(0,self.number_of_contents-1),tuple(generate_random_cache_state(self.cache_size,state_info)))
                next_state=self.all_states[next_state_info]

            action.append(next_state)
        else:
            #Exploit learned values
            i=1
            state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state)
            top_k = np.argpartition(self.q_table[state], -self.number_of_contents)[-10:]
            action_index = top_k[np.argsort(self.q_table[state][top_k])[-i]]
            next_state_info=action_format_pi(self.cache_size,state_info,action_index)
            next_state=self.all_states[next_state_info]
            while(check_action(self.cache_states,state,self.env.id_to_cache,next_state)==False):
                i+=1
                action_index=np.argsort(self.q_table[state])[-i]
                next_state_info=action_format_pi(self.cache_size,state_info,action_index)
                next_state=self.all_states[next_state_info]


            action.append(next_state)
        return action


    def learn(self,state,action,reward,next_state):
        new_value=0
        old_value=0
        for act in action:
            action_index=find_action_index_from_states(self.cache_states,state,act,self.cache_size,self.env.id_to_cache)
            old_value=self.q_table[state,action_index]
            next_state_max=np.max(self.q_table[next_state])
            new_value=(1-self.learning_rate)*old_value + self.learning_rate * ( reward + self.gamma * next_state_max)
            self.q_table[state,action_index] = new_value
        return new_value,old_value


    def eps_decay(self,episode):
        self.eps=self.eps_min +(self.eps_init-self.eps_min)*math.exp(-episode*self.eps_decay_rate)
        return


## Q-Learning Update Function

In [ ]:

def update(env,agent,max_iter = 1000,plot_interval=1000,threshold = 0.1,eps_decay=False,train_enable=True):

    costs = []
    episodes=0
    counter=0
    max_difference=threshold+1
    start_time = time.time()

    list_of_rewards = []
    list_of_penalties = []
    if( not train_enable):
       eps_decay=False
       agent.eps=0


    for episodes in trange(max_iter):
        new_values=[]
        old_values=[]
        # old_q_table = copy.copy(agent.q_table)
        state = env.refresh()



        i = 0

        done = False

        current_penalty = 0
        current_reward = 0

        while (not done):
            action=agent.choose_actions(state)

            next_state,reward,done=env.simulate(action,state)


            if(train_enable):
                new_value,old_value=agent.learn(state,action,reward,next_state)
                new_values.append(new_value)
                old_values.append(old_value)




            state=next_state

            i+=1
            current_reward+=reward

            if reward == env.rewards[0]:
                current_penalty+=1

        episodes+=1



        if(train_enable):
          cost=np.square(np.subtract(new_values,old_values)).mean()
          costs.append(cost)

        if(i!=0):
            list_of_penalties.append(current_penalty/i)
            list_of_rewards.append(current_reward/i)


        if eps_decay:
            agent.eps_decay(episodes)

        if(episodes % plot_interval == 0):

            _periodic_plot(episodes,plot_interval,list_of_rewards,costs=costs,
                           extra_lists=None, extra_titles=None, episode_slices=500,
                           threshold=threshold,early_stopage=False)
            print(agent.eps)
    running_time=time.time()-start_time

    return running_time,episodes,costs,list_of_rewards,list_of_penalties






# DQN Model

## DQN Architecture

In [ ]:
class DQNFlex(nn.Module):
    def __init__(self, num_features, cache_size, num_contents, nn_layers=[64, 32], dropout_rate=0.1,rec_only=False):
        super(DQNFlex, self).__init__()
        self.rec_only=rec_only
        self.num_actions = num_contents * (cache_size + 1)
        self.num_features = num_features
        self.proj_dim=16


        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.proj=nn.Linear(num_features,self.proj_dim)

        self.attention = nn.MultiheadAttention(embed_dim=self.proj_dim, num_heads=4, batch_first=True)
        self.norm=nn.LayerNorm(self.proj_dim)

        layers = []
        in_features = self.proj_dim
        for hidden_size in nn_layers:
            layers.append(nn.Linear(in_features, hidden_size))
            # layers.append(nn.LayerNorm(hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_features = hidden_size

        self.shared_layers = nn.Sequential(*layers)

        if self.rec_only==False:
            self.cache_out = nn.Linear(nn_layers[-1], cache_size + 1)
        self.rec = nn.Linear(nn_layers[-1], 1)

    def forward(self, x):
        if len(x.shape) == 2:
            x = x.unsqueeze(0)

        batch_size, num_contents, _ = x.size()


        x=self.proj(x)


        atten_x, _ = self.attention(x, x, x)
        x = self.norm(atten_x + x)


        x = self.shared_layers(x)

        rec_output = self.rec(x)

        if self.rec_only:
            return rec_output.view(batch_size,-1)

        cache_output = self.cache_out(x)

        alpha=torch.sigmoid(self.alpha).to(cache_output.dtype)


        # rec_q_values = (rec_output - rec_output.mean()) / (rec_output.std() + 1e-6)
        # cache_q_values = (cache_output - cache_output.mean()) / (cache_output.std() + 1e-6)

        output = torch.lerp(cache_output, rec_output.expand_as(cache_output), alpha).view(batch_size, -1)


        return output

## Experience Replay


In [ ]:

class ExperienceReplay:

    def __init__(self,capacity):
        self.capacity=capacity
        self.memory = collections.deque(maxlen=self.capacity)

    def __len__(self):
        return len(self.memory)

    def append(self,experience):
        self.memory.append(experience)

    def sample(self, batch_size):
        # Randomly sample a batch of experiences from the replay buffer
        experiences = random.sample(self.memory, batch_size)

        states, actions, rewards, dones, next_states, states_rep, next_states_rep, state_topN , state_topN_map, next_topN, next_topN_map = zip(*[
            (e.state, e.action, e.reward, e.done, e.next_state, e.state_rep, e.next_state_rep,e.state_topN,e.state_topN_map,e.next_topN,e.next_topN_map)
            for e in experiences
        ])

        return states, actions, rewards, dones, next_states, states_rep, next_states_rep, state_topN , state_topN_map ,next_topN, next_topN_map



## Prioritized Experience Replay using Sum Tree

In [ ]:
class SumTree:

    def __init__(self,capacity):
        self.capacity= capacity
        self.tree=[0] * (2 * self.capacity - 1)
        self.data=[None] * self.capacity
        self.write_idx=0
        self.num_entries=0

    def total(self):
        return self.tree[0]

    def update(self,data_idx,priority):
        idx=data_idx+self.capacity-1
        difference=priority-self.tree[idx]
        self.tree[idx]=priority

        parent=(idx-1) // 2
        while parent>=0:
            self.tree[parent]+=difference
            parent=(parent-1) // 2


    def add(self,priority,data):
        self.data[self.write_idx]=data
        self.update(self.write_idx,priority)

        self.write_idx = (self.write_idx+1) % self.capacity
        self.num_entries = min(self.capacity,self.num_entries+1)

    def get(self,cumsum):
        assert cumsum <=self.total()

        idx=0
        while 2*idx +1 < len(self.tree):
            left,right = 2*idx + 1  , 2*idx + 2

            if cumsum <=self.tree[left]:
                idx=left
            else:
                idx= right
                cumsum=cumsum - self.tree[left]

        data_idx=idx-self.capacity + 1

        return data_idx,self.tree[idx],self.data[data_idx]


class PrioritizedExperienceReplay:
    def __init__(self, capacity,device, eps=1e-2, alpha=0.7, beta=0.4):
        self.capacity = capacity
        self.tree = SumTree(capacity)
        self.memory = [None]*self.capacity
        self.alpha = alpha
        self.beta = beta
        self.max_priority = eps
        self.eps = eps
        self.max_beta = 1.0
        self.device = device
        self.pin_memory = self.device.type == "cuda"
        self.non_blocking = self.pin_memory


    def __len__(self):
        return self.tree.num_entries

    def append(self, experience):
        # Add experience to memory and SumTree
        self.memory[self.tree.write_idx] = experience
        self.tree.add(self.max_priority, self.tree.write_idx)

    def sample(self, batch_size):
        sample_idxs, tree_idxs = [], []
        priorities = np.empty([batch_size, 1], dtype=np.float32)
        segment = self.tree.total() / batch_size

        for i in range(batch_size):
            a, b = segment * i, segment * (i + 1)
            cumsum = random.uniform(a, b)
            tree_idx, priority, sample_idx = self.tree.get(cumsum)

            priorities[i] = priority
            tree_idxs.append(tree_idx)
            sample_idxs.append(sample_idx)

        probs = priorities / self.tree.total()
        weights = (self.tree.num_entries * probs) ** -self.beta
        weights = weights / weights.max()

        experiences = [self.memory[i] for i in sample_idxs]
        rewards, dones, states_rep, next_states_rep = [], [], [], [],
        states_content,states_cache,next_states_content,next_states_cache=[],[],[],[]
        action_content,action_cache=[],[]
        state_topN,state_topN_map,next_topN,next_topN_map=[],[],[],[]
        for exp in experiences:
            states_content.append(exp.state[0])
            states_cache.append(exp.state[1])
            next_states_content.append(exp.next_state[0])
            next_states_cache.append(exp.next_state[1])
            action_content.append(exp.action[0][0])
            action_cache.append(exp.action[0][1])
            rewards.append(exp.reward)
            dones.append(exp.done)
            states_rep.append(exp.state_rep)
            next_states_rep.append(exp.next_state_rep)
            state_topN.append(exp.state_topN)
            state_topN_map.append(exp.state_topN_map)
            next_topN.append(exp.next_topN)
            next_topN_map.append(exp.next_topN_map)


        if isinstance(states_rep, list) and all(isinstance(s, torch.Tensor) for s in states_rep):
              states_batch_rep = torch.stack([s.to(self.device,non_blocking=self.non_blocking) for s in states_rep])

        if isinstance(next_states_rep, list) and all(isinstance(ns, torch.Tensor) for ns in next_states_rep):
              next_states_batch_rep = torch.stack([ns.to(self.device,non_blocking=self.non_blocking) for ns in next_states_rep])

        
        states_content = torch.as_tensor(states_content, dtype=torch.int32)
        states_cache = torch.as_tensor(states_cache, dtype=torch.int32)
        action_content = torch.as_tensor(action_content, dtype=torch.int32)
        action_cache = torch.as_tensor(action_cache, dtype=torch.int32)
        rewards = torch.as_tensor(rewards, dtype=torch.float32).to(device=self.device,non_blocking=True)
        dones = torch.as_tensor(dones, dtype=torch.int32).to(device=self.device,non_blocking=True)
        next_states_content = torch.as_tensor(next_states_content, dtype=torch.int32)#.to(device=self.device,non_blocking=True)
        next_states_cache = torch.as_tensor(next_states_cache, dtype=torch.int32)#.to(device=self.device,non_blocking=True)
        weights = torch.as_tensor(weights, dtype=torch.float32).to(device=self.device,non_blocking=True)

        return (states_content,states_cache),(action_content,action_cache), rewards, dones, (next_states_content,next_states_cache), states_batch_rep, next_states_batch_rep, tree_idxs, weights,state_topN,state_topN_map,next_topN,next_topN_map

    def update_priority(self, indices, priorities):
        for data_idx, priority in zip(indices, priorities):
            priority = priority ** self.alpha
            self.tree.update(data_idx, priority)
            self.max_priority = max(self.max_priority, priority)

    def beta_increase(self, increment):
        if self.beta + increment <= self.max_beta:
            self.beta += increment
        else:
            self.beta = self.max_beta




# Double DQN Agent

In [ ]:
class DQNAgent:

    def __init__(self,env,state_features=3,hidden_dim=64,hidden_layers=5,learning_rate=0.1,gamma=0.9,eps=0.9,max_iter=50000,replay_type="PER",replay_buffer_size=10000,batch_size=64,taf=0.01,nn_layers=[40,20],rec_only=False):
            self.env=env
            self.hidden_dim=hidden_dim
            self.hidden_layers=hidden_layers
            self.learning_rate=learning_rate
            self.gamma=gamma
            self.max_iter=max_iter
            self.replay_type=replay_type
            self.replay_buffer_size=replay_buffer_size
            self.batch_size=batch_size
            self.taf=taf
            self.rec_only=rec_only
            
            self.state_features=state_features
            self.device=torch.device("cuda" if torch.cuda.is_available() else "cpu")


            # Recommended
            self.recommended_mask = torch.zeros(
                (self.env.number_of_contents,self.env.number_of_contents), dtype=torch.bool, device=self.device
            )

            for c, rec_set in enumerate(self.env.recommended):
                indices = torch.tensor(list(rec_set), dtype=torch.long)
                self.recommended_mask[c, indices] = True

            self.training_episodes=0
            self.training_episodes_output=0
            self.training_episodes_feature=0

            self.decay_counter=0
            self.decay_counter_output=0
            self.decay_counter_feature=0

            self.train_output_layer=False

            self.rec_num=1000
            self.rec_options_num=self.rec_num if self.rec_num< self.env.number_of_contents else self.env.number_of_contents

            self.checkpoint=f'{self.__class__}_{self.state_features}.chkpt'
            self.output_layer_checkpoint=f'model_wc_{self.state_features}_c{self.rec_options_num}_c{self.env.cache_size}.chkpt'
            self.use_baseline=False

            self.nn_layers=nn_layers
            self.scaler = torch.amp.GradScaler(self.device.type)



            self.cache_loss=[]
            self.rec_loss=[]


            # must be changed depending on the state representation
            self.state_size=self.state_features*self.env.number_of_contents
            #############################

            self.u_states=self.create_u_states()
            self.popularity=torch.from_numpy(self.env.popularity).to(device=self.device)


        # E-greedy parameters
            self.eps_init=eps
            self.eps=self.eps_init
            self.eps_decay_rate=1e-4
            self.eps_decay_rate_cache=1e-4
            self.eps_min=0.05

        # Policy and Target networks
            self.q_network_policy=DQNFlex(self.state_features,self.env.cache_size,self.rec_options_num,nn_layers=self.nn_layers,rec_only=self.rec_only).to(self.device)
            self.q_network_target=DQNFlex(self.state_features,self.env.cache_size,self.rec_options_num,nn_layers=self.nn_layers,rec_only=self.rec_only).to(self.device)
            self.q_network_target.load_state_dict(self.q_network_policy.state_dict())
            self.q_network_target.eval()
            for param in self.q_network_target.parameters():
                param.requires_grad = False
        # Optimizer for the network
            self.optimizer=optim.AdamW(self.q_network_policy.parameters(),lr=self.learning_rate,weight_decay=1e-3)
            self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='min', factor=0.5, patience=2000, min_lr=1e-5)

            self.loss=nn.MSELoss(reduction='none')


        #Experience Replay
            if self.replay_type=="PER":
                self.memory = PrioritizedExperienceReplay(self.replay_buffer_size,self.device,alpha=0.7,beta=0.4)
                self.beta_increment=1/(max_iter*0.9)
            else:
                self.memory = ExperienceReplay(self.replay_buffer_size)

            self.cache_hit=torch.zeros(2,self.env.number_of_contents,device=self.device)
            self.popularity=torch.from_numpy(self.env.popularity).to(device=self.device)

    def create_u_states(self):
        u_states=np.zeros([self.env.number_of_contents,self.rec_options_num],dtype=np.int64)
        u_states = np.argsort(self.env.u, axis=1)[:, ::-1][:, :self.rec_options_num]
        return u_states


    def find_topN(self,state):
        
        essential = list(state[1])
        if state[0] not in essential:
            essential.append(state[0])
        remaining = self.rec_options_num - len(essential)
        if remaining > 0:
            candidates = np.setdiff1d(self.u_states[state[0]], essential,assume_unique=True)
            essential.extend(candidates[:remaining].tolist())

        topN_map = {item: idx for idx, item in enumerate(essential)}
        return essential,topN_map



    def representation_transformation(self,state):
        if self.state_features==2:
            return self.representation_transformation_2(state)
        elif self.state_features==4:
            return self.representation_transformation_4(state)
        else:
            return self.representation_transformation_3(state)



    def representation_transformation_2(self,state):
        """
        Representing the state as a tensor  of size [number of contents,2] where first column has info about the recommended states and second column has info about the cached states

        state : current state , e.g  1,(0,2,4,9)


        """
        topN,topN_map=self.find_topN(state)
        topN_t = torch.tensor(topN, dtype=torch.long, device=self.device)
        recommended_tensor = self.recommended_mask[state[0]][topN_t].float()
        cache_t = torch.zeros(self.env.number_of_contents, device=self.device)
        cache_t[state[1]] = 1.0
        cache_tensor = cache_t[topN_t]

        return torch.cat((recommended_tensor.view(-1,1),cache_tensor.view(-1,1)), 1),topN,topN_map


    def representation_transformation_3(self, state):
        """
        State representation with 3 features .
        1st Feature: A tensor with the recommended content.
        2nd Feature: A tensor with the cached content.
        3rd Feature: A tensor with the sum of the cached and recommended states of the next state.
        """

        topN,topN_map=self.find_topN(state)
        topN_t = torch.tensor(topN, dtype=torch.long, device=self.device)
        recommended_tensor = self.recommended_mask[state[0]][topN_t].float()

        cache_state = state[1]
        cache_t = torch.zeros(self.env.number_of_contents, device=self.device)
        cache_t[list(cache_state)] = 1.0
        cache_tensor = cache_t[topN_t]

        third_feature = torch.zeros(1,self.rec_options_num, device=self.device)


        third_feature[0]=torch.div(self.cache_hit[0][topN_t],torch.clamp(self.cache_hit[1][topN_t],min=1))
        third_feature=normalize_tensor(third_feature)

        return torch.cat((recommended_tensor.view(-1,1),cache_tensor.view(-1,1), third_feature.view(-1,1)), 1),topN,topN_map


    def representation_transformation_4(self, state):
        """
        State representation with 3 features .
        1st Feature: A tensor with the recommended content.
        2nd Feature: A tensor with the cached content.
        3rd Feature: A tensor with the sum of the cached and recommended states of the next state.
        """

        topN,topN_map=self.find_topN(state)
        topN_t = torch.tensor(topN, dtype=torch.long, device=self.device)
        recommended_tensor = self.recommended_mask[state[0]][topN_t].float()

        cache_state = state[1]
        cache_t = torch.zeros(self.env.number_of_contents, device=self.device)
        cache_t[list(cache_state)] = 1.0
        cache_tensor = cache_t[topN_t]

        third_feature = torch.zeros(1,self.rec_options_num, device=self.device)
        fourth_feature = torch.zeros(1,self.rec_options_num, device=self.device)

        third_feature[0]=torch.div(self.cache_hit[0][topN_t],torch.clamp(self.cache_hit[1][topN_t],min=1))
        third_feature=normalize_tensor(third_feature)


        fourth_feature[0]=torch.div(self.popularity[topN_t],torch.max(self.popularity[topN_t]))
        fourth_feature=normalize_tensor(fourth_feature)


        return torch.cat((recommended_tensor.view(-1,1),cache_tensor.view(-1,1), third_feature.view(-1,1),fourth_feature.view(-1,1)), 1),topN,topN_map


    def eps_decay(self,episode):
        self.eps=self.eps_min +(self.eps_init-self.eps_min)*math.exp(-episode*self.eps_decay_rate)
        return



    def update_target_network(self):
        """
        Used for soft updating the target network.
        """
        with torch.no_grad():
            for p_target, p_policy in zip(self.q_network_target.parameters(), self.q_network_policy.parameters()):
                p_target.data.mul_(1 - self.taf)
                p_target.data.add_(self.taf * p_policy.data)


    def save_model(self,path):
        """
        Saves the policy network,tagrget network , the value of epsilon and the experience buffer in a file named model.chkpt .
        """

        torch.save({
                'epsilon':self.eps,
                'q_network_policy':self.q_network_policy.state_dict(),
                'q_network_target':self.q_network_target.state_dict(),
                'training_episodes':self.training_episodes_feature,
                'decay_counter':self.decay_counter_feature,
                'optimizer':self.optimizer.state_dict()


        },os.path.join(path,self.checkpoint))


    def load_model(self,path):
        """
        Loads the policy network,tagrget network , the value of epsilon and the experience buffer from a file named model.chkpt .
        """
        checkpoint=torch.load(os.path.join(path,self.checkpoint),self.device)
        self.q_network_policy.load_state_dict(checkpoint['q_network_policy'])
        self.q_network_target.load_state_dict(checkpoint['q_network_target'])
        self.eps=checkpoint['epsilon']
        self.training_episodes=checkpoint['training_episodes']
        self.decay_counter=checkpoint['decay_counter']
        self.optimizer=checkpoint['optimizer']






## DQN Agent without caching

In [ ]:
class DQNAgent_NC(DQNAgent):

    def __init__(self,env,state_features=3,hidden_dim=64,hidden_layers=5,learning_rate=0.1,gamma=0.9,eps=0.9,max_iter=50000,replay_type="PER",replay_buffer_size=10000,batch_size=64,taf=0.01,nn_layers=[40,20],rec_only=True):
        super().__init__(env,state_features,hidden_dim,hidden_layers,learning_rate,gamma,eps,max_iter,replay_type,replay_buffer_size,batch_size,taf,nn_layers,rec_only)
        self.number_of_actions=self.rec_options_num
        
        

    def choose_actions(self, state,state_rep,topN,topN_map):
        """
        RL recommendations plus popularity based caching.

        """

        action = []
        
        if random.uniform(0, 1) < self.eps and self.use_baseline==True:  # Imitate baseline (early training)
            cache=state[1]
            i=1
            if self.env.popularity[state[0]]>np.min(self.env.popularity[state[1]]) and state[0] not in state[1]:
                index=np.argmin(self.env.popularity[state[1]])
                cache=generate_cache_state(self.env.cache_size,state,index)

            next_item=state[1][np.argsort(self.env.u[state[0]][state[1]])[-i]]
            next_state=next_item,cache
            while(not check_action_DQN(state,next_state)):
                i+=1
                next_item=state[1][np.argsort(self.env.u[state[0]][state[1]])[-i]]
                next_state=next_item,cache
            if check_action_DQN(state, next_state):
                action.append(next_state)
            else:
                print("ERROR")


        else:
            if random.uniform(0,1) < self.eps :# Random Exploration
                valid_rec = [c for c in topN if c != state[0]]  # all valid rec choices
                next_content = random.choice(valid_rec)

                cache=state[1]
                pop_in_cache = self.popularity[list(state[1])]
                min_idx = int(torch.argmin(pop_in_cache).item())
                if self.popularity[state[0]] > pop_in_cache[min_idx] and state[0] not in state[1]:
                    cache = generate_cache_state(self.env.cache_size, state, min_idx)

                next_state=next_content,cache
                action.append(next_state)


            else:  # Use learned Q-network (later training)



                with torch.no_grad():

                    q_network_output=self.q_network_policy(state_rep)
                    current_state=topN_map[state[0]]

                    mask_actions = torch.zeros(1,self.number_of_actions, dtype=torch.bool,device=self.device)

                    mask_actions[0][current_state]=True



                    q_masked = torch.where(mask_actions, torch.tensor([-1e8], device=self.device), q_network_output.to(self.device))
                    chosen_action = int(torch.argmax(q_masked).item())

                    cache=state[1]
                    next_state_rec=topN[chosen_action]

                    if self.env.popularity[state[0]]>np.min(self.env.popularity[state[1]]) and state[0] not in state[1]:
                        index=np.argmin(self.env.popularity[state[1]])
                        cache=generate_cache_state(self.env.cache_size,state,index)

                    next_state=next_state_rec,cache
                    action.append(next_state)


        return action


    def learn(self, experiences):
            if self.memory.__class__ == PrioritizedExperienceReplay:
                _ , actions, rewards, dones, next_states, states_batch, next_states_batch, idx, weights ,_, topN_map,_,next_topN_map = experiences
            else:
                _ , actions, rewards, dones, next_states, states_batch, next_states_batch, _, topN_map, _,next_topN_map = experiences


            # Start autocast for mixed precision during forward and loss computation
            with  torch.autocast(device_type=self.device.type):
                policy_action = self.q_network_policy(states_batch)
                predicted_action = torch.zeros(self.batch_size, 1, device=self.device)

                action_contents = actions[0].tolist()

                action_indices = []
                for top_map,a in zip(topN_map,action_contents):
                    action_indices.append(top_map[a]) 

                predicted_action = policy_action[torch.arange(self.batch_size), torch.tensor(action_indices, device=self.device)]



                with torch.no_grad():
                    next_actions= self.q_network_policy(next_states_batch)
                    mask_actions = torch.zeros_like(next_actions, dtype=torch.bool, device=self.device)

                    next_contents = next_states[0].tolist()   # one transfer
                    

                    for i in range(self.batch_size):
                        next_state_index = next_topN_map[i][next_contents[i]] 
                        mask_actions[i][next_state_index] = True

                    q_masked = torch.where(~mask_actions, next_actions, torch.tensor([-1e8]).to(self.device))
                    best_actions = q_masked.argmax(1, keepdim=True)

                    target_q_action=self.q_network_target(next_states_batch)

                    target_q=target_q_action.gather(1,best_actions)



                target_action = rewards.view(self.batch_size,-1)+ self.gamma * target_q.view(self.batch_size, -1)


                loss = self.loss(predicted_action.view(-1), target_action.view(-1))



                if self.memory.__class__ == PrioritizedExperienceReplay:
                    td_errors = torch.sub(target_action, predicted_action)
                    new_priorities = torch.add(torch.abs(td_errors), self.memory.eps)
                    self.memory.update_priority(idx, chain.from_iterable(new_priorities.tolist()))
                    self.memory.beta_increase(self.beta_increment)
                    loss = torch.mean(weights.squeeze()* loss)



                self.optimizer.zero_grad()
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step(loss.detach())


                return loss.item()



## DQN Agent with caching

In [ ]:
class DQNAgent_WC(DQNAgent):

    def __init__(self,env,state_features=3,hidden_dim=64,hidden_layers=5,learning_rate=0.1,gamma=0.9,eps=0.9,max_iter=50000,replay_type="PER",replay_buffer_size=10000,batch_size=64,taf=0.01,nn_layers=[40,20],rec_only=False):
        super().__init__(env,state_features,hidden_dim,hidden_layers,learning_rate,gamma,eps,max_iter,replay_type,replay_buffer_size,batch_size,taf,nn_layers,rec_only)
        self.number_of_actions=self.rec_options_num*(self.env.cache_size+1)
        


    def choose_actions(self, state,state_rep,topN,topN_map):
        action = []
        



        if random.uniform(0, 1) < self.eps and self.use_baseline==True:  # Imitate baseline (early training)
                cache=state[1]
                i=1
                if self.env.popularity[state[0]]>np.min(self.env.popularity[state[1]]) and state[0] not in state[1]:
                    index=np.argmin(self.env.popularity[state[1]])
                    cache=generate_cache_state(self.env.cache_size,state,index)

                next_item=state[1][np.argsort(self.env.u[state[0]][state[1]])[-i]]
                next_state=next_item,cache
                while(not check_action_DQN(state,next_state)):
                    i+=1
                    next_item=state[1][np.argsort(self.env.u[state[0]][state[1]])[-i]]
                    next_state=next_item,cache
                if check_action_DQN(state, next_state):
                    action.append(next_state)
                else:
                    print("ERROR")


        else:
            if random.uniform(0,1) < self.eps :# Random Exploration
                valid_rec = [c for c in topN if c != state[0]]  # all valid rec choices
                next_content = random.choice(valid_rec)

                next_state = next_content, generate_random_cache_state(self.env.cache_size,state)          
                action.append(next_state)

            else:  # Use learned Q-network (later training)


                with torch.no_grad():

                    q_network_output=self.q_network_policy(state_rep)
                    current_state=topN_map[state[0]]

                    mask_actions = torch.zeros(1,self.number_of_actions, dtype=torch.bool,device=self.device)

                    mask_actions[0][range(current_state*(self.env.cache_size+1), (current_state+1)*(self.env.cache_size+1))] = True
                    if state[0] in state[1]:
                        for k in range(self.env.cache_size):
                            mask_actions[0][range(k,self.number_of_actions,(self.env.cache_size+1))]= True


                    q_masked = torch.where(mask_actions, torch.tensor([-1e8], device=self.device), q_network_output.to(self.device))
                    chosen_action = torch.argmax(q_masked).item()

                    next_state=action_format(self.env.cache_size,state,chosen_action,topN)
                    action.append(next_state)


        return action


    def learn(self, experiences):
            if self.memory.__class__ == PrioritizedExperienceReplay:
                _, actions, rewards, dones, next_states, states_batch, next_states_batch, idx, weights, _ , topN_map, _,topN_map_next = experiences
            else:
                _ , actions, rewards, dones, next_states, states_batch, next_states_batch, _, topN_map, _, topN_map_next = experiences


          # Start autocast for mixed precision during forward and loss computation
            with  torch.autocast(device_type=self.device.type):
                policy_action = self.q_network_policy(states_batch)
                predicted_action = torch.zeros(self.batch_size, 1, device=self.device)

               
                action_contents = actions[0].tolist()
                

                action_indices = []
                for top_map,a in zip(topN_map,action_contents):
                    action_indices.append(top_map[a]) 

                predicted_action = policy_action[torch.arange(self.batch_size), torch.tensor(action_indices, device=self.device)]



                with torch.no_grad():
                    next_actions= self.q_network_policy(next_states_batch)
                    mask_actions = torch.zeros_like(next_actions, dtype=torch.bool, device=self.device)

                  
                    next_contents = next_states[0].tolist()
                    next_caches = next_states[1].tolist()

                    for i in range(self.batch_size):
                        next_state_index = topN_map_next[i][next_contents[i]]  
                        mask_actions[i][next_state_index * (self.env.cache_size + 1) : (next_state_index + 1) * (self.env.cache_size + 1)] = True
                        if next_contents[i] in next_caches[i]:
                            for k in range(self.env.cache_size):
                                mask_actions[i][k::(self.env.cache_size + 1)] = True

                    q_masked = torch.where(~mask_actions, next_actions, torch.tensor([-1e8]).to(self.device))
                    best_actions = q_masked.argmax(1, keepdim=True)

                    target_q_action=self.q_network_target(next_states_batch)

                    target_q=target_q_action.gather(1,best_actions)



                target_action = rewards.view(self.batch_size,-1)+ self.gamma * target_q.view(self.batch_size, -1)


                loss = self.loss(predicted_action.view(-1), target_action.view(-1))



                if self.memory.__class__ == PrioritizedExperienceReplay:
                    td_errors = torch.sub(target_action, predicted_action)
                    new_priorities = torch.add(torch.abs(td_errors), self.memory.eps)
                    self.memory.update_priority(idx, chain.from_iterable(new_priorities.tolist()))
                    self.memory.beta_increase(self.beta_increment)
                    loss = torch.mean(weights.squeeze()* loss)



                self.optimizer.zero_grad()
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step(loss.detach())


                return loss.item()






# DQN Train Function

In [ ]:
def get_current_lr(optimizer):

    for param_group in optimizer.param_groups:

        return param_group['lr']

def train(env,agent,max_iter = 1000,plot_interval=5000,threshold = 1e-3,eps_decay=False,path=None,train_enable=True,save_load=False):

    costs = []
    episodes=0
    max_difference=threshold+1
    start_time = time.time()

    list_of_rewards = []
    list_of_penalties = []
    list_of_recommendation_rewards=[]
    list_of_caching_rewards=[]
    if(path is not None and os.path.isfile(path+agent.checkpoint) and save_load==True):
        agent.load_model(path)
        print(agent.training_episodes)
        print("Checkpoint loaded!")

    agent.q_network_policy.train()




    if( not train_enable):
       eps_decay=False
       agent.eps=0
       agent.use_baseline=False
       agent.q_network_policy.eval()


    for episodes in trange(max_iter):

        state = env.refresh()
        i = 0

        done = False

        current_penalty = 0
        current_reward = 0
        current_loss=0
        current_recommendation_reward=0
        current_caching_reward=0




        state_rep=None
        next_state_rep=None



        while (not done):


            if next_state_rep is None:
              state_rep,topN,topN_map=agent.representation_transformation(state)
            else:
              state_rep=next_state_rep.clone()
              topN=next_topN.copy()
              topN_map=next_topN_map.copy()



            action=agent.choose_actions(state,state_rep,topN,topN_map)
            


            next_state,reward,done=env.simulate(action,state)

            if(train_enable==True):

                if isinstance(agent,DQNAgent_WC) and agent.state_features>2:
                    agent.cache_hit[0]*=0.999
                    agent.cache_hit[1]*=0.999
                    if next_state[0] in state[1]:
                        agent.cache_hit[0,next_state[0]]+=1
                    agent.cache_hit[1,next_state[0]]+=1


                next_state_rep,next_topN,next_topN_map=agent.representation_transformation(next_state)
                exp=Experience(state,action,reward,done,next_state,state_rep,next_state_rep,topN,topN_map,next_topN,next_topN_map)
                agent.memory.append(exp)

            if action[0][0]==next_state[0]:
                current_recommendation_reward+=1
            else:
                current_recommendation_reward+=0

            if action[0][0] in state[1]:
              current_caching_reward+=1
            else:
              current_caching_reward+=0

            state=next_state

            i+=1
            current_reward+=reward
            if reward == env.rewards[0]:
                current_penalty+=1

        
        
        if train_enable:
            if agent.memory.__len__()>=agent.batch_size:
                experiences=agent.memory.sample(agent.batch_size)
                current_loss=agent.learn(experiences)
                agent.update_target_network()

        
            agent.training_episodes+=1


        episodes+=1

        if(i!=0):
            costs.append(current_loss)
            list_of_penalties.append(current_penalty/i)
            list_of_rewards.append(current_reward/i)
            list_of_recommendation_rewards.append(current_recommendation_reward/i)
            list_of_caching_rewards.append(current_caching_reward/i)

        if(eps_decay and train_enable==True):
            agent.eps_decay(agent.training_episodes)


        if(eps_decay and np.mean(list_of_rewards[episodes-1000:episodes]) < current_reward and train_enable==True):
            agent.decay_counter+=1


        if(episodes % plot_interval == 0):

            converged =_periodic_plot(episodes,plot_interval,list_of_rewards,costs=costs,
                           extra_lists=[list_of_recommendation_rewards,list_of_caching_rewards],
                           extra_titles=["Recommendation Acceptance Rate","Recommending Cached Content Rate"],
                           episode_slices=500,threshold=threshold,early_stopage=train_enable)

            print("eps= ",agent.eps)

            
            if(save_load==True and train_enable==True):
                agent.save_model(path)

            if converged:
                break




    running_time=time.time()-start_time
    if train_enable==True:
        finalize_training(agent)

    return running_time,episodes,costs,list_of_rewards,list_of_penalties






# Non RL Impementation

### Baseline

In [ ]:
class Non_RL_agent_baseline:

     def __init__(self,env):
        self.env=env
        self.top_popular = set(np.argpartition(self.env.popularity, -self.env.cache_size)[-self.env.cache_size:])


     def choose_actions(self,state):
        action=[]
        cache=state[1]
        i=0
        if state[0] in self.top_popular and state[0] not in state[1]:
          index = np.argmin(self.env.popularity[state[1]])
          cache=generate_cache_state(self.env.cache_size,state,index)

        next_item_choice=np.argmax(self.env.u[state[0]])
        next_state=next_item_choice,cache

        assert check_action_DQN(state,next_state), "Illegal action"


        action.append(next_state)

        return action


### Greedy

In [ ]:
class Non_RL_agent_greedy:

     def __init__(self,env):
        self.env=env
        self.top_popular = set(np.argpartition(self.env.popularity, -self.env.cache_size)[-self.env.cache_size:])

     def choose_actions(self,state):
        action=[]
        cache=state[1]
        i=0
        if state[0] in self.top_popular and state[0] not in state[1]:
          index = np.argmin(self.env.popularity[state[1]])
          cache=generate_cache_state(self.env.cache_size,state,index)
        next_item_choices = np.argsort(self.env.u[state[0]][list(state[1])])[::-1]

        next_item=state[1][next_item_choices[i]]
        next_state=next_item,cache

        while check_action_DQN(state,next_state)==False :
            i+=1
            next_item=state[1][next_item_choices[i]]
            next_state=next_item,cache



        action.append(next_state)

        return action


## Udpate

In [ ]:
def train_non_RL(env,agent,max_iter = 1000,plot_interval=2000,threshold = 1e-3,eps_decay=False,path=None,train_enable=True,save_load=False):


    episodes=0

    start_time = time.time()

    list_of_rewards = []
    list_of_penalties = []
    list_of_recommendation_rewards=[]
    list_of_caching_rewards=[]


    for episodes in trange(max_iter):

        state = env.refresh()
        i = 0

        done = False

        current_penalty = 0
        current_reward = 0
        current_loss=0
        current_recommendation_reward=0
        current_caching_reward=0

        while (not done):

            action=agent.choose_actions(state)

            next_state,reward,done=env.simulate(action,state)

            state=next_state

            i+=1
            current_reward+=reward


            if reward == env.rewards[0]:
                current_penalty+=1

            if action[0][0]==next_state[0]:
                current_recommendation_reward+=1
            else:
                current_recommendation_reward+=0

            if next_state[0] in state[1]:
              current_caching_reward+=1
            else:
              current_caching_reward+=0


        episodes+=1

        if(i!=0):
            list_of_penalties.append(current_penalty/i)
            list_of_rewards.append(current_reward/i)
            list_of_recommendation_rewards.append(current_recommendation_reward/i)
            list_of_caching_rewards.append(current_caching_reward/i)



        if(episodes % plot_interval == 0):

            _periodic_plot(episodes,plot_interval,list_of_rewards,costs=None,
                           extra_lists=[list_of_recommendation_rewards,list_of_caching_rewards],
                           extra_titles=["Recommendation Acceptance Rate","Recommending Cached Content Rate"],
                           episode_slices=500,threshold=threshold,early_stopage=False)



    running_time=time.time()-start_time
    return running_time,episodes,list_of_rewards,list_of_penalties

# Testing


In [ ]:
list_of_results=[]
pre_trained_results=[]

popularity_file='popularity_file.npy'
dataset='ratings_small.csv'
path=""
u_file='u_file.npy'
set_seed(31)


In [ ]:
number_of_contents=500
cache_size=5
rewards=[0,1]
number_of_recommendations=1
max_iter=10000
testing_max_iter=5000
eps_value=1
eps_decay=True
corr_threshold=0.85
gamma=0.95
converge_threshold=0
prob_to_leave=0.05



if(os.path.isfile(u_file)):
    u=np.load(u_file)


    if(u.shape[1]!=number_of_contents):
        if(os.path.isfile(dataset)):
            u=create_u(number_of_contents,dataset)
        else:
            u=create_u(number_of_contents)
        print("U file created!")
    else:
        print("U file loaded!")

else:
    if(os.path.isfile(dataset)):
        u=create_u(number_of_contents,dataset)
    else:
        u=create_u(number_of_contents)

    print("U file created!")




if(os.path.isfile(popularity_file)):
    popularity=np.load(popularity_file)


    if(popularity.__len__()!=number_of_contents):
        if(os.path.isfile(dataset)):
            popularity=create_popularity(number_of_contents,dataset)
        else:
            popularity=create_popularity(number_of_contents,uniform=False)
        print("Popolarity file created!")
    else:
        print("Popolarity file loaded!")

else:
    if(os.path.isfile(dataset)):
        popularity=create_popularity(number_of_contents,dataset)
    else:
        popularity=create_popularity(number_of_contents,uniform=False)


    print("Popolarity file created!")




#same performance // threshold = 0.5 // scenario 2 // high correlation for cached states
# u[0]=[0,0.3,0.4,0.2]
# u[1]=[0.7,0,0.8,0.4]
# u[2]=[0.6,0.8,0,0.7]
# u[3]=[0.2,0.3,0.4,0]

# caching agent better perfomance // threshold = 0.5 // scenario 1 // low correlation for cached states
# u[0]=[0,0.7,0.4,0.8]
# u[1]=[0.25,0,0.3,0.4]
# u[2]=[0.25,0.25,0,0.4]
# u[3]=[0.6,0.8,0.7,0]


# u[0]=[0,0.3,0.6,0.2]
# u[1]=[0.7,0,0.4,0.1]
# u[2]=[0.4,0.2,0,0.8]
# u[3]=[0.3,0.9,0.4,0]

# u[0]=[0,0.15,0.65,0.07]
# u[1]=[0.53,0,0.05,0.50]
# u[2]=[0.03,0.43,0,0.09]
# u[3]=[0.42,0.82,0.12,0]


# np.save(u_file,u)
# np.save(popularity_file,popularity)
print(u.mean())
# print(u)
# print(popularity)

## Non RL Testing

### Baseline

In [ ]:

non_RL_env = Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold, prob_to_leave,user_type='quality_aware',dataset=None, u =u,popularity = popularity)
Agent = Non_RL_agent_baseline(non_RL_env)
running_time,episodes,list_of_rewards,list_of_penalties = train_non_RL(non_RL_env,Agent,max_iter=testing_max_iter)
costs=[]
res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
pre_trained_results.append(res)

### Greedy

In [ ]:

non_RL_env = Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold, prob_to_leave,user_type='quality_aware',dataset=None, u =u,popularity = popularity)
Agent = Non_RL_agent_greedy(non_RL_env)
running_time,episodes,list_of_rewards,list_of_penalties = train_non_RL(non_RL_env,Agent,max_iter=testing_max_iter)
costs=[]
res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
pre_trained_results.append(res)

## Policy Iteration Testing

In [ ]:
# PI_env=Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold,1-gamma,user_type='quality_aware',u=u,popularity=popularity,tabular=True)
# PI_agent=Agent_PI(PI_env,number_of_contents,cache_size,corr_threshold,u,gamma=gamma,max_iter=10000,threshold=1e-6,popularity=popularity)
# running_time,episodes,list_of_rewards,list_of_penalties=update_PI(PI_env,PI_agent,max_iter=testing_max_iter)
# costs=[]
# res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
# pre_trained_results.append(res)


## TABULAR TESTING

### Tabular without caching

In [ ]:
# env_tab_1=Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold,1-gamma,user_type='quality_aware',u=u,tabular=True)
# RL_1=Agent_without_caching(env_tab_1,learning_rate=0.01,gamma=gamma,max_iter=max_iter,eps=eps_value)
# running_time,episodes,costs,list_of_rewards,list_of_penalties=update(env_tab_1,RL_1,max_iter=max_iter,eps_decay=eps_decay,threshold=converge_threshold,train_enable=True)
# res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
# list_of_results.append(res)


In [ ]:
# running_time,episodes,costs,list_of_rewards,list_of_penalties=update(env_tab_1,RL_1,max_iter=testing_max_iter,eps_decay=eps_decay,threshold=converge_threshold,train_enable=False)
# res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
# pre_trained_results.append(res)

### Tabular with caching

In [ ]:
# env_tab_2=Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold,1-gamma,user_type='quality_aware',u=u,tabular=True)
# RL_2=Agent_with_caching(env_tab_2,learning_rate=0.01,gamma=gamma,max_iter=max_iter,eps=eps_value)
# running_time,episodes,costs,list_of_rewards,list_of_penalties=update(env_tab_2,RL_2,max_iter=max_iter,eps_decay=eps_decay,threshold=converge_threshold,train_enable=True)
# res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
# list_of_results.append(res)


In [ ]:
# running_time,episodes,costs,list_of_rewards,list_of_penalties=update(env_tab_2,RL_2,max_iter=testing_max_iter,eps_decay=eps_decay,threshold=converge_threshold,train_enable=False)
# res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
# pre_trained_results.append(res)


## DQN TEST

### DQN without caching

In [ ]:
batch_size=32
max_iter=50000
nn_layers=[120,100,80]

env_DQN_NC=Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold,1-gamma,user_type='quality_aware',popularity=popularity,u=u)

RL_DQN_NC=DQNAgent_NC(env_DQN_NC,state_features=4,hidden_dim=1000,hidden_layers=3,learning_rate=1e-3,gamma=gamma,eps=eps_value,max_iter=max_iter,replay_type="PER",batch_size=batch_size,replay_buffer_size=10_000,taf=0.01,nn_layers=nn_layers)


running_time,episodes,costs,list_of_rewards,list_of_penalties=train(env_DQN_NC,RL_DQN_NC,max_iter=max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=True,save_load=False)
res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
list_of_results.append(res)


running_time,episodes,costs,list_of_rewards,list_of_penalties=train(env_DQN_NC,RL_DQN_NC,max_iter=testing_max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=False,save_load=False)
res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
pre_trained_results.append(res)




### DQN with caching

In [ ]:
batch_size=32
max_iter=50000
nn_layers=[120,100,80]

env_DQN_WC=Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold,1-gamma,user_type='quality_aware',popularity=popularity,u=u)

RL_DQN_WC=DQNAgent_WC(env_DQN_WC,state_features=4,hidden_dim=1000,hidden_layers=3,learning_rate=1e-3,gamma=gamma,eps=eps_value,max_iter=max_iter,replay_type="PER",batch_size=batch_size,replay_buffer_size=20_000,taf=0.01,nn_layers=nn_layers)


running_time,episodes,costs,list_of_rewards,list_of_penalties=train(env_DQN_WC,RL_DQN_WC,max_iter=max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=True,save_load=False)
res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
list_of_results.append(res)


running_time,episodes,costs,list_of_rewards,list_of_penalties=train(env_DQN_WC,RL_DQN_WC,max_iter=testing_max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=False,save_load=False)
res=Result(list_of_rewards,list_of_penalties,costs,episodes,running_time)
pre_trained_results.append(res)




## Results

In [ ]:
episode_slices=1000

label=None
label=["RL Recommendation, popularity  based Caching","RL joint Recommendation  and Caching"]
# label=["DDQN"]
graphic_compare_results_reward(list_of_results,eps_decay,"Cache Hit Rate",label=label,episode_slices=episode_slices,file_name="rewards.png")

graphic_compare_results_cost(list_of_results,eps_decay,"Loss",label=label,episode_slices=episode_slices,file_name="loss.png")

label=["baseline","greedy","RL \n Recom/tion \n popularity \n based Caching","RL joint \n Recom/tion \n and Caching"]
# label=["baseline","greedy","RL Recommendation, popularity  based Caching","RL joint Recommendation  and Caching"]
graphic_compare_results_reward(pre_trained_results,eps_decay,"Cache Hit Rate",label=label,episode_slices=500,file_name="pre_trained_reward.png")


for i in range(len(list_of_results)):
    print(np.mean(list_of_results[i].reward),np.mean(list_of_results[i].penalty),np.mean(list_of_results[i].cost),list_of_results[i].time//60)


for i in range(len(pre_trained_results)):
    print(np.mean(pre_trained_results[i].reward),np.mean(pre_trained_results[i].cost))

# for j in range(100,0,-1) :
#     print(RL_DQN_WC.memory.memory[j])

### Bars Plot

In [ ]:

label=["baseline","greedy","RL \n Recom/tion \n popularity \n based Caching","RL joint \n Recom/tion \n and Caching"]

agents_names=label
values=[]

for i in pre_trained_results:
  values.append(100*np.mean(i.reward))
colors = ['blue','orange', 'green','red' ]
# Create the bar graph
plt.bar(agents_names,values,color=colors)

for i, (name, value) in enumerate(zip(agents_names, values)):
  plt.annotate(f"{int(value)}", xy=(i, value), ha="center", va="bottom")  # Adjust ha/va if needed

plt.ylim((0,100))
plt.xlabel('Agent Type')
plt.ylabel('Cache hit rate %')
plt.title('Comparison of Agent Performance')
plt.savefig('pre_trained_bars.png')
plt.show()

In [ ]:
# policy_wo_cahing=[]
# for i in range(len(env_tab_1.all_states)):
#     state=find_state_by_id_faster(env_tab_1.cache_states,i)
#     next_state=np.argsort(RL_1.q_table[i])[-1],state[1]
#     print(state,"--->",next_state)
#     policy_wo_cahing.append((state,"--->",next_state))



In [ ]:
# policy_caching=[]
# print(u)
# for i in range(len(env_tab_2.all_states)):
#     state=find_state_by_id_faster(env_tab_2.cache_states,i)
#     next_state=action_format(cache_size,state,np.argsort(RL_2.q_table[i])[-1])
#     print(state,"--->",next_state,np.max(RL_2.q_table[i]))
#     policy_caching.append((state,"----->",next_state))


# PI vs Q-learning

In [ ]:

# policy_caching=[]
# for i in range(len(env_tab_2.all_states)):
#     state=find_state_by_id_faster(env_tab_2.cache_states,i)
#     next_state=action_format_pi(cache_size,state,np.argmax(RL_2.q_table[i]))
#     next_state_id=env_tab_2.all_states[next_state]
#     policy_caching.append((state,"----->",next_state,"-----",PI_agent.value[next_state_id]))



# counter=0
# number=0
# for i in range(len(env_tab_2.all_states)):
#     if(policy_caching[i][2]!=action_format_pi(PI_agent.cache_size,find_state_by_id_faster(PI_agent.cache_states,i),PI_agent.policy[i])):
#         next_state=action_format_pi(PI_agent.cache_size,find_state_by_id_faster(PI_agent.cache_states,i),PI_agent.policy[i])
#         id=env_tab_2.all_states[next_state]
#         number+=1
#         print(policy_caching[i],"-------------------",next_state,PI_agent.value[id])
#         if(policy_caching[i][4]!=PI_agent.value[id]):
#             counter+=1

# print(env_tab_2.number_of_states-number,"/",env_tab_2.number_of_states)
# print(env_tab_2.number_of_states-counter,"/",env_tab_2.number_of_states)
